In [1]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain youtube-transcript-api pytube


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from getpass import getpass

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = getpass("Enter LangSmith API key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-from-scratch"

In [2]:

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())


/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_17707/2878383631.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:

print("Total documents:", len(docs))
print("First document preview:")
print(docs[0].page_content[:500])


Total documents: 2
First document preview:






LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: 


In [4]:

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.2", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "Summarize the following document concisely:\n\n{doc}"
)

summarize_chain = prompt | llm | StrOutputParser()

# Start with the first document
summary = summarize_chain.invoke({
    "doc": docs[0].page_content[:6000]
})

print(summary)


The document discusses the concept of LLM (Large Language Model) powered autonomous agents, which utilize LLM as the agent's brain, complemented by several key components. The components include:

1. Planning: Breaking down large tasks into smaller subgoals, enabling efficient handling of complex tasks. This can be done through task decomposition, self-reflection, and refinement.
2. Memory: Short-term and long-term memory, which provide the agent with the capability to retain and recall information over extended periods.
3. Tool use: The agent learns to call external APIs for extra information, including current information, code execution capability, and access to proprietary information sources.

The document also discusses various approaches to planning, including:

* Chain of thought (CoT) and Tree of Thoughts (Yao et al. 2023), which transform big tasks into multiple manageable tasks.
* LLM+P (Liu et al. 2023), which relies on an external classical planner to do long-horizon plann

In [5]:

# Generate summaries for both documents
summaries = []

for i, doc in enumerate(docs):
    print(f"Summarizing document {i + 1}...")

    summary = summarize_chain.invoke({
        "doc": doc.page_content[:6000]
    })

    summaries.append(summary)

print("Total summaries:", len(summaries))
print("\nFirst summary:\n", summaries[0])


Summarizing document 1...
Summarizing document 2...
Total summaries: 2

First summary:
 Here is a concise summary of the document:

The document discusses the concept of LLM (Large Language Model) powered autonomous agents, which use LLM as the brain and complement it with several key components: planning, memory, and tool use.

**Planning:**

* Task decomposition: breaking down large tasks into smaller subgoals using techniques like Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023).
* Self-reflection: allowing agents to learn from mistakes and refine past action decisions.

**Memory:**

* Short-term memory: utilizing in-context learning to learn from the model's weights.
* Long-term memory: retaining and recalling information over extended periods using an external vector store and fast retrieval.

**Tool Use:**

* Learning to call external APIs for extra information, such as current information, code execution capability, and access to proprietary information sources.

Th

In [12]:

import uuid

from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore

# 1. Initialize local embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Create vector store for summaries
vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=embeddings
)

# 3. Create storage for original documents
store = InMemoryByteStore()
id_key = "doc_id"

# 4. Initialize MultiVectorRetriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key
)

# 5. Generate unique IDs for documents
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 6. Link each summary to its original document
summary_docs = [
    Document(
        page_content=s,
        metadata={id_key: doc_ids[i]}
    )
    for i, s in enumerate(summaries)
]

# 7. Store summaries and original documents
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

print("Summaries and original documents stored successfully!")


Summaries and original documents stored successfully!


In [7]:

query = "Memory in agents"

# 1. Search the vector store directly
summary_results = vectorstore.similarity_search(query, k=2)

print("=== Retrieved Summaries ===")
for i, doc in enumerate(summary_results):
    print(f"\nSummary {i+1}:")
    print(doc.page_content[:500])
    print("Document ID:", doc.metadata["doc_id"])

# 2. Retrieve original documents
retrieved_docs = retriever.invoke(query)

print("\n=== Retrieved Original Documents ===")
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print("Title:", doc.metadata.get("title", "N/A"))
    print("Source:", doc.metadata.get("source", "N/A"))
    print("Content length:", len(doc.page_content))


=== Retrieved Summaries ===

Summary 1:
Here is a concise summary of the document:

The document discusses the concept of LLM (Large Language Model) powered autonomous agents, which use LLM as the brain and complement it with several key components: planning, memory, and tool use.

**Planning:**

* Task decomposition: breaking down large tasks into smaller subgoals using techniques like Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023).
* Self-reflection: allowing agents to learn from mistakes and refine past action decisi
Document ID: 0a1dde72-b26f-4810-98d7-87936c3f5dec

Summary 2:
The document "Thinking about High-Quality Human Data" by Lilian Weng discusses the importance of high-quality human data in modern data deep learning model training. The author highlights two directions to approach high data quality:

1. **Human Raters ↔ Data Quality**: This involves collecting human data through a series of steps, including task design, selecting and training raters, collectin

## ColBERT

In [8]:
! pip install -U ragatouille

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 8.2 MB/s  0:00:17m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.3/876.3 kB 8.0 MB/s  0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any

In [1]:
from ragatouille import RAGPretrainedModel

print("RAGatouille imported successfully!")

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_23001/640297814.py:1: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel
/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/aish/Desktop/rag_from_scratc

RAGatouille imported successfully!


In [2]:
RAG = RAGPretrainedModel.from_pretrained(
    "colbert-ir/colbertv2.0"
)


[Nov 11, 23:39:08] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


In [3]:

import requests

def get_wikipedia_page(title: str):
    URL = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    headers = {
        "User-Agent": "RAGatouille_tutorial/0.0.1 (ben@clavie.eu)"
    }

    response = requests.get(
        URL, params=params, headers=headers, timeout=30
    )
    response.raise_for_status()

    data = response.json()
    page = next(iter(data["query"]["pages"].values()))

    return page.get("extract")

full_document = get_wikipedia_page("Hayao_Miyazaki")

print("Document loaded:", full_document is not None)

if full_document:
    print("Document length:", len(full_document))
    print("Preview:", full_document[:500])


Document loaded: True
Document length: 71431
Preview: Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the greatest and most accomplished filmmakers in the history of animation.
Born in Tokyo City, Miyazaki expressed intere


In [4]:

RAG.index(
    collection=[full_document],
    index_name="Miyazaki-123",
    max_document_length=180,
    split_documents=True,
)


---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[Nov 11, 23:40:59] #> Creating directory .ragatouille/colbert/indexes/Miyazaki-123 


[Nov 11, 23:41:01] [0] 		 #> Encoding 128 passages..


  0%|          | 0/4 [00:00<?, ?it/s]/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
100%|██████████| 4/4 [00:04<00:00,  1.14s/it]

[Nov 11, 23:41:06] [0] 		 avg_doclen_est = 130.953125 	 len(local_sample) = 128
[Nov 11, 23:41:06] [0] 		 Creating 2,048 partitions.
[Nov 11, 23:41:06] [0] 		 *Estimated* 16,762 embeddings.
[Nov 11, 23:41:06] [0] 		 #> Saving the indexing plan to .ragatouille/colbert/indexes/Miyazaki-123/plan.json ..


used 15 iterations (0.8604s) to cluster 15924 items into 2048 clusters
[0.035, 0.036, 0.037, 0.031, 0.031, 0.034, 0.031, 0.033, 0.03, 0.033, 0.031, 0.035, 0.032, 0.033, 0.032, 0.034, 0.03, 0.029, 0.032, 0.034, 0.033, 0.034, 0.032, 0.035, 0.034, 0.029, 0.035, 0.032, 0.033, 0.032, 0.034, 0.036, 0.037, 0.031, 0.033, 0.032, 0.032, 0.034, 0.033, 0.036, 0.033, 0.034, 0.031, 0.03, 0.032, 0.031, 0.031, 0.034, 0.034, 0.03, 0.032, 0.033, 0.032, 0.032, 0.032, 0.03, 0.034, 0.035, 0.037, 0.03, 0.03, 0.032, 0.033, 0.033, 0.033, 0.032, 0.033, 0.034, 0.032, 0.03, 0.033, 0.032, 0.032, 0.032, 0.033, 0.031, 0.032, 0.033, 0.033, 0.032, 0.033, 0.035, 0.03, 0.035, 0.03, 0.031, 0.034, 0.033, 0.03, 0.039, 0.033, 0.035, 0.032, 0.033, 0.031, 0.031, 0.037, 0.032, 0.035, 0.033, 0.037, 0.037, 0.033, 0.032, 0.035, 0.032, 0.032, 0.031, 0.034, 0.03, 0.034, 0.033, 0.032, 0.031, 0.033, 0.034, 0.033, 0.033, 0.034, 0.036, 0.03, 0.03, 0.031, 0.033, 0.03, 0.035, 0.032, 0.032]


0it [00:00, ?it/s]

[Nov 11, 23:41:07] [0] 		 #> Encoding 128 passages..


100%|██████████| 4/4 [00:04<00:00,  1.14s/it]
1it [00:04,  4.64s/it]
100%|██████████| 1/1 [00:00<00:00, 2165.36it/s]

[Nov 11, 23:41:12] #> Optimizing IVF to store map from centroids to list of pids..
[Nov 11, 23:41:12] #> Building the emb2pid mapping..
[Nov 11, 23:41:12] len(emb2pid) = 16762



100%|██████████| 2048/2048 [00:00<00:00, 216764.27it/s]

[Nov 11, 23:41:12] #> Saved optimized IVF to .ragatouille/colbert/indexes/Miyazaki-123/ivf.pid.pt
Done indexing!


'.ragatouille/colbert/indexes/Miyazaki-123'

In [5]:
results = RAG.search(
    query="What animation studio did Miyazaki found?",
    k=3
)

results

Loading searcher for index Miyazaki-123 for the first time... This may take a few seconds
[Nov 11, 23:41:50] #> Loading codec...
[Nov 11, 23:41:50] #> Loading IVF...
[Nov 11, 23:41:50] Loading segmented_lookup_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Nov 11, 23:41:57] #> Loading doclens...


100%|██████████| 1/1 [00:00<00:00, 1714.76it/s]

[Nov 11, 23:41:57] #> Loading codes and residuals...



100%|██████████| 1/1 [00:00<00:00, 317.46it/s]

[Nov 11, 23:41:57] Loading filter_pids_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


[Nov 11, 23:42:03] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: What animation studio did Miyazaki found?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2054,  7284,  2996,  2106,  2771,  3148, 18637,  2179,
         1029,   102,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103])
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])



/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


[{'content': '=== Studio Ghibli ===\n\n\n==== Foundation and Laputa (1985–1987) ====\n\nFollowing the success of Nausicaä of the Valley of the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15, 1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio\'s name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309 aircraft, meaning "a hot wind that blows in the desert" in Italian.',
  'score': 25.698028564453125,
  'rank': 1,
  'document_id': '98d9c92c-9add-4c1d-a497-47803ee7006d',
  'passage_id': 42},
 {'content': 'Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is 

In [6]:

results = RAG.search(
    query="What animation studio did Miyazaki found?",
    k=3
)

for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print("Score:", result["score"])
    print("Content:", result["content"][:500])



--- Result 1 ---
Score: 25.698028564453125
Content: === Studio Ghibli ===


==== Foundation and Laputa (1985–1987) ====

Following the success of Nausicaä of the Valley of the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15, 1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio's name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309 aircraft, meaning "a hot wind that blows in the desert" in Italian.

--- Result 2 ---
Score: 25.541723251342773
Content: Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the greatest and most acco

In [7]:

# Convert ColBERT into a LangChain retriever
retriever = RAG.as_langchain_retriever(k=3)

# Retrieve relevant documents
retrieved_docs = retriever.invoke(
    "What animation studio did Miyazaki found?"
)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print(doc.page_content[:500])



--- Document 1 ---
=== Studio Ghibli ===


==== Foundation and Laputa (1985–1987) ====

Following the success of Nausicaä of the Valley of the Wind, Miyazaki and Takahata founded the animation production company Studio Ghibli on June 15, 1985, as a subsidiary of Tokuma Shoten, with offices in Kichijōji designed by Miyazaki. The studio's name had been registered a year earlier; Miyazaki named it after the nickname of the Caproni Ca.309 aircraft, meaning "a hot wind that blows in the desert" in Italian.

--- Document 2 ---
Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the greatest and most accomplished filmmakers in the history of animation.
Born in Tokyo Cit

/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/Users/aish/Desktop/rag_from_scratch/.venv-colbert/lib/python3.11/site-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
